In [ ]:

# Step 0: Install & Imports

!pip install librosa kagglehub --quiet

import os
import numpy as np
import librosa
import kagglehub
from scipy.special import logsumexp
from sklearn.utils import shuffle  # For random shuffling


# Step 1: Download Dataset

path = kagglehub.dataset_download("kongaevans/speaker-recognition-dataset")
print("Dataset path:", path)

main_data_folder = os.listdir(path)[0]
speaker_root_dir = os.path.join(path, main_data_folder)

# List speaker folders
speakers = [s for s in os.listdir(speaker_root_dir) if os.path.isdir(os.path.join(speaker_root_dir, s))]
print("Number of speakers found:", len(speakers))
print("Sample speakers:", speakers[:5])

# Step 2: Feature Extraction

def extract_features(file_path, n_mfcc=13):
    try:
        signal, sr = librosa.load(file_path, sr=None)
        mfccs = librosa.feature.mfcc(y=signal, sr=sr, n_mfcc=n_mfcc)
        return mfccs.T  # frames x features
    except Exception as e:
        print(f"Error loading {file_path}: {e}")
        return None

# Step 3: Load Data (subset)

def load_data(dataset_path, max_speakers=10, files_per_speaker=50):
    data = {}

    main_data_folder = os.listdir(dataset_path)[0]
    actual_speaker_root = os.path.join(dataset_path, main_data_folder)

    selected_speakers = [s for s in os.listdir(actual_speaker_root)
                         if os.path.isdir(os.path.join(actual_speaker_root, s))][:max_speakers]

    for speaker in selected_speakers:
        folder = os.path.join(actual_speaker_root, speaker)
        files = [f for f in os.listdir(folder) if f.endswith(".wav")][:files_per_speaker]

        features = []
        for file in files:
            file_path = os.path.join(folder, file)
            mfcc = extract_features(file_path)
            if mfcc is not None:
                features.append(mfcc)

        if features:
            data[speaker] = features

    return data

data = load_data(path)
print("\nLoaded Data:")
for s in data:
    print(s, "->", len(data[s]), "files")

# Step 4: Train/Test Split

def split_data(data, train_ratio=0.7):
    train_data = {}
    test_data = {}

    for speaker in data:
        files = shuffle(data[speaker], random_state=42)  # shuffle before splitting
        split = int(len(files) * train_ratio)
        train_data[speaker] = files[:split]
        test_data[speaker] = files[split:]

    return train_data, test_data

train_data, test_data = split_data(data)
print("\nTrain/Test split:")
for s in train_data:
    print(s, "-> train:", len(train_data[s]), "| test:", len(test_data[s]))

# Step 5: GMM (EM Algorithm)

def initialize_gmm(X, K=4):
    N, D = X.shape
    means = X[np.random.choice(N, K, replace=False)]
    covs = [np.eye(D) for _ in range(K)]
    weights = np.ones(K) / K
    return means, covs, weights

def log_gaussian_pdf(x, mean, cov):
    D = len(mean)
    reg_cov = cov + 1e-6 * np.eye(D)

    try:
        sign, logdet = np.linalg.slogdet(reg_cov)
        if sign <= 0:
            return -np.inf
        cov_inv = np.linalg.inv(reg_cov)
    except np.linalg.LinAlgError:
        return -np.inf

    diff = x - mean
    return -0.5 * (D * np.log(2 * np.pi) + logdet + diff @ cov_inv @ diff.T)

def e_step(X, means, covs, weights):
    N = X.shape[0]
    K = len(weights)
    log_responsibilities = np.zeros((N, K))

    for k in range(K):
        log_prob_X_given_k = np.array([log_gaussian_pdf(X[n], means[k], covs[k]) for n in range(N)])
        log_responsibilities[:, k] = np.log(weights[k] + 1e-300) + log_prob_X_given_k

    log_denominators = logsumexp(log_responsibilities, axis=1, keepdims=True)
    gamma = np.exp(log_responsibilities - log_denominators)
    return gamma

def m_step(X, gamma):
    N, D = X.shape
    K = gamma.shape[1]
    Nk = gamma.sum(axis=0)
    means = np.zeros((K, D))
    covs = []
    weights = Nk / N

    for k in range(K):
        if Nk[k] > 1e-6:
            means[k] = (gamma[:, k][:, None] * X).sum(axis=0) / Nk[k]
            diff = X - means[k]
            cov = (gamma[:, k][:, None] * diff).T @ diff / Nk[k]
            covs.append(cov + 1e-6 * np.eye(D))
        else:
            means[k] = X[np.random.choice(N, 1)]
            covs.append(1e-6 * np.eye(D))
    weights = np.clip(weights, 1e-300, None)
    weights /= weights.sum()
    return means, covs, weights

def train_gmm(X, K=4, iterations=15):
    means, covs, weights = initialize_gmm(X, K)
    for i in range(iterations):
        gamma = e_step(X, means, covs, weights)
        means, covs, weights = m_step(X, gamma)
    return means, covs, weights

# Step 6: Train Models
models = {}
print("\nTraining GMMs...")
for speaker in train_data:
    X = np.vstack(train_data[speaker])
    print(f"Training {speaker} with {X.shape[0]} frames...")
    models[speaker] = train_gmm(X)
print("Training complete.")

# Step 7: Likelihood + Prediction

def compute_likelihood(X, model):
    means, covs, weights = model
    total_log_likelihood = 0
    for x in X:
        log_probs = np.array([np.log(weights[k]+1e-300) + log_gaussian_pdf(x, means[k], covs[k])
                              for k in range(len(weights))])
        total_log_likelihood += logsumexp(log_probs)
    return total_log_likelihood

def predict(models, X):
    scores = {speaker: compute_likelihood(X, models[speaker]) for speaker in models}
    return max(scores, key=scores.get), scores

# Step 8: Evaluation

correct = 0
total = 0

print("\nTesting...")
for speaker in test_data:
    for sample in test_data[speaker]:
        pred, scores = predict(models, sample)
        print(f"True: {speaker}, Pred: {pred}")
        if pred == speaker:
            correct += 1
        total += 1

accuracy = correct / total
print(f"\nFinal Accuracy: {accuracy * 100:.2f}%")

100%|██████████| 231M/231M [00:01<00:00, 155MB/s]

Extracting files...


Dataset path: /root/.cache/kagglehub/datasets/kongaevans/speaker-recognition-dataset/versions/1
Number of speakers found: 7
Sample speakers: ['Jens_Stoltenberg', 'Nelson_Mandela', 'Julia_Gillard', 'other', 'Magaret_Tarcher']

Loaded Data:
Jens_Stoltenberg -> 50 files
Nelson_Mandela -> 50 files
Julia_Gillard -> 50 files
other -> 2 files
Magaret_Tarcher -> 50 files
_background_noise_ -> 4 files
Benjamin_Netanyau -> 50 files

Train/Test split:
Jens_Stoltenberg -> train: 35 | test: 15
Nelson_Mandela -> train: 35 | test: 15
Julia_Gillard -> train: 35 | test: 15
other -> train: 1 | test: 1
Magaret_Tarcher -> train: 35 | test: 15
_background_noise_ -> train: 2 | test: 2
Benjamin_Netanyau -> train: 35 | test: 15

Training GMMs...
Training Jens_Stoltenberg with 1120 frames...
Training Nelson_Mandela with 1120 frames...
Training Julia_Gillard with 1120 frames...
Training other with 2584 frames...
Training Magaret_Tarcher with 1120 frames...
Training _background_noise_ with 4038 frames...
Trainin